# Step 2 practice: build the labels yourself

The real code is `ml/labels.py` (already written and checked). Here you rebuild its core ideas by hand, one small piece at a time, and every piece has a self-check.

**What a label is.** At each 15-minute moment, the label answers a question about the near future, for example "will a low start within the next 60 minutes?". The model never sees it as an input; it is the answer it learns to predict.

| Piece | What you build |
|---|---|
| 1 | Fix the sensor reading (offset correction) |
| 2 | Find where an episode starts |
| 3 | Label one recording by hand (low within 60 min) |
| 4 | Compare with `labels.py`, then the whole dataset |
| 5 | Questions |

In [ ]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from labels import load, sensor_offsets, build_labels, episode_starts, LOW

ts = load("../data/shanghai_t2dm_clean/shanghai_t2dm_timeseries.csv")
rid = "2083_0_20201214"                      # a recording with many finger-prick checks
one = ts[ts["recording_id"] == rid]
print(len(one), "rows")

## 1. Fix the sensor reading (offset correction)

Some sensors read lower or higher than a finger-prick blood test. For each recording we measure the typical gap and subtract it before deciding what counts as a low. **Labels use the corrected value; the model's inputs stay raw.**

In [ ]:
off = sensor_offsets(ts)            # one number per recording: sensor minus finger-prick, typically negative
print("offset of", rid, "=", round(off[rid], 1), "mg/dL")
print("dataset median offset =", round(float(np.median(list(off.values()))), 1))

**Your turn.** Recordings `2077_0_20210803` and `2083_0_20201214` have very different offsets. Print both. Which sensor reads too low, and what would happen to its "lows" after correction (more, or fewer)?

In [ ]:
# Your turn: write your code here

## 2. Find where an episode starts

A low **episode** is 2 or more readings in a row below 70. We mark only the **first** reading of each episode, so one long low counts once.

Write `my_starts(v, below, thr)` that returns a True/False array: True at the first reading of every run of 2+ readings below (or above) `thr`. Hints:
- `x = v < thr` is True where a reading is beyond the line.
- A run of 2 or more means `x[i]` and `x[i+1]` are both True.
- It is a *start* if the previous reading was not True.

In [ ]:
def my_starts(v, below, thr):
    # Your turn: write the function
    pass

In [ ]:
toy = np.array([80, 65, 60, 75, 68, 66, 90, 50, 55, 50])   # expected starts at positions 1, 4 and 7
print(np.where(my_starts(toy, True, 70))[0])
assert list(np.where(my_starts(toy, True, 70))[0]) == [1, 4, 7], "not yet right"
assert (my_starts(toy, True, 70) == episode_starts(toy, True, 70)).all()
print("OK: matches labels.py")

## 3. Label one recording by hand: "a low starts within the next 60 minutes"

At moment `i`: the corrected reading must be **70 or more** (we do not warn about something already happening), the next 5 readings must exist, and a low episode must **start** within the next 4 readings (60 minutes). The grid below puts the recording on a clean 15-minute clock; gaps stay empty (NaN), never filled.

In [ ]:
s = one.set_index("timestamp")["cgm_mgdl"].resample("15min", origin="start").mean()
v = s.values - off[rid]                         # corrected readings
n = len(v)
st = my_starts(v, True, LOW)                    # starts of low episodes (from piece 2)
print(n, "moments on the grid;", int(st.sum()), "low episodes start")

In [ ]:
low = np.full(n, np.nan)       # NaN = cannot decide, True/False otherwise
# Your turn: fill low[i] for every moment i using the rules above
print(int(np.nansum(low)), "moments are followed by a low within 60 min")

## 4. Compare with `labels.py`

`labels.py` also blanks the low label in the first 12 hours of a recording (the sensor is still settling; decision 33). Compare after that point.

In [ ]:
lib = build_labels(one).set_index("timestamp")
mine = pd.Series(low, index=s.index, name="mine")
cmp = pd.concat([mine, lib["low60"]], axis=1, join="inner").dropna()
print(len(cmp), "moments compared;", int((cmp["mine"].astype(bool) != cmp["low60"].astype(bool)).sum()), "differences")

In [ ]:
# the first 12 h in the library are blank on purpose; everything after must match
late = cmp[cmp.index >= s.index[0] + pd.Timedelta(hours=12)]
assert (late["mine"].astype(bool) == late["low60"].astype(bool)).all(), "your labels differ from labels.py"
print("OK: your hand-made labels equal labels.py")

In [ ]:
labels = build_labels(ts)
for col in ["low60", "spike120", "large_rise120"]:
    y = labels[col].dropna().astype(bool)
    print(f"{col:14s} moments {len(y):6d}  positives {int(y.sum()):6d}  ({100 * y.mean():.1f}%)")
# expected: 106100 / 378, 84716 / 13795, 92877 / 13977

**Your turn** (add cells below as needed)
1. `spike120` works like `low60`, but above 180, looking 8 readings (2 hours) ahead, and the current reading must be 180 or less. Change your loop to build it for this recording and compare with `lib["spike120"]`.
2. `large_rise120` uses a personal bar instead of a fixed number. Read the comments in `labels.py` (`build_labels`) and explain in your own words why `shift(1)` is used before the rolling window.

In [ ]:
# Your turn: write your code here

## 5. Questions

1. Why do we subtract the offset only for the labels and not for the model's inputs?
2. Why do we mark only the **first** reading of an episode as the start?
3. Why must the current reading be at least 70 for the low label?
4. Why is the low label blank in the first 12 hours?
5. Why is each recording placed on its own 15-minute grid instead of using the raw timestamps?

**Send back to Claude:** your answers and anything that surprised you.